# Runtime Polymorphism and Abstract Classes


```{contents}
:local:
:depth: 2
```


```{index} Distinguish Choosing a Member from Dispatching Its Implementation
```



```{index} Distinguish Choosing a Member from Dispatching Its Implementation
```

## Distinguish Choosing a Member from Dispatching Its Implementation



A variable's compile-time type determines which members a call can name and participates in overload resolution. A virtual call can then dispatch to an override selected from the runtime object. These are different decisions. Overloading adds a signature choice; overriding changes implementation of an inherited virtual contract. Hiding a nonvirtual member with new is another behavior and should not be described as runtime overriding.

| Situation | What determines the important choice? |
| --- | --- |
| Overloaded method candidates | Compile-time argument/type rules |
| Override of a virtual member | Runtime object for the chosen virtual call |
| Hidden nonvirtual member | The member selected through the reference's static type |

Trace a base-typed reference holding a derived policy object. For a virtual Calculate call, follow the override implemented by that object. Now change only the variable declaration to the derived type: the stored object has not changed, although more members may become available to the caller. A cast is not a request to construct a new object or change the object into another subtype.

An abstract class can leave some behavior unimplemented while supplying common state or concrete operations. It cannot be directly instantiated as a completed object. A concrete derived type must provide the required behavior before clients can use an instance. The requirement belongs to the type contract, not to a convention that an unimplemented method should return zero.

Guided trace: use two derived policies through a sequence of base references. Predict which implementation each virtual call reaches and check that every result satisfies the shared domain/result conditions. Then compare the preserved hiding/override variants in the supporting reading; use them to explain why matching method names alone do not prove identical dispatch.

For related explanation, see the further reading below.[^development-reading]


A client should not need a separate calculation branch for every discount class. Runtime polymorphism lets one call use the implementation associated with the actual object. The declared type of a variable describes the operations available to the client; the runtime object determines which override runs.

Keep the object diagram from the previous section. A variable declared `DiscountPolicy` can refer to a percentage or fixed policy. Changing the variable's declared type does not replace the object or strip away its derived behavior.


```{index} Virtual Calls Select an Override
```

## Virtual Calls Select an Override




```{index} Virtual Calls Select an Override
```


A base method marked `virtual` allows a derived class to override it. The derived method uses `override` with the matching signature. In the loop below, every variable has the same declared base type, but each object's override provides its calculation.

Before running, identify the three actual object types and calculate their results independently. There is no cast or type-name comparison in the client loop.


In [1]:
using System;

DiscountPolicy[] policies = {
    new DiscountPolicy(), new PercentageDiscount(0.10m), new FixedDiscount(5m)
};
foreach (DiscountPolicy policy in policies)
    Console.WriteLine(policy.Apply(100m).ToString("F2"));

class DiscountPolicy
{
    public virtual decimal Apply(decimal subtotal)
    {
        if (subtotal < 0m) throw new ArgumentOutOfRangeException(nameof(subtotal));
        return subtotal;
    }
}

class PercentageDiscount : DiscountPolicy
{
    private readonly decimal rate;
    public PercentageDiscount(decimal rate)
    {
        if (rate < 0m || rate > 1m) throw new ArgumentOutOfRangeException(nameof(rate));
        this.rate = rate;
    }
    public override decimal Apply(decimal subtotal)
    {
        decimal validSubtotal = base.Apply(subtotal);
        return validSubtotal * (1m - rate);
    }
}

class FixedDiscount : DiscountPolicy
{
    private readonly decimal reduction;
    public FixedDiscount(decimal reduction)
    {
        if (reduction < 0m) throw new ArgumentOutOfRangeException(nameof(reduction));
        this.reduction = reduction;
    }
    public override decimal Apply(decimal subtotal)
    {
        decimal validSubtotal = base.Apply(subtotal);
        return Math.Max(0m, validSubtotal - reduction);
    }
}


100.00
90.00
95.00


The output is 100.00, 90.00, and 95.00. If a fourth policy preserves the same contract, this loop does not need another branch. That is the useful extension point: clients depend on the behavior available through the general type.

A nonvirtual method is not overridden by merely declaring another method with the same name. C# also supports member hiding, but hiding follows different selection rules and is outside this core sequence. Use explicit `override` when the intended behavior is runtime replacement of a virtual member.


```{index} Overriding Is Different from Overloading
```

## Overriding Is Different from Overloading




```{index} Overriding Is Different from Overloading
```


**Overriding** supplies a derived implementation for an inherited virtual or abstract member. **Overloading** gives methods the same name with different parameter lists. Overload selection uses the call's available type and argument information; it does not provide the same extension mechanism as a virtual call.

Keep the distinction concrete: `Apply(decimal subtotal)` overridden in two policy classes is one polymorphic operation. `Print(string)` and `Print(int)` are two overloads selected by the argument type. Changing only a return type is not enough to define a distinct overload.

An overridden `ToString` is another familiar virtual operation. It should return a useful text representation rather than perform printing as a side effect. Formatting an object and changing its business state are separate responsibilities.


For the original numeric/string overloads, nonvirtual-versus-virtual animal comparison, and the base/sealed-override shape variations, continue with [OOP examples and variations](1204_oop_examples.ipynb). The current policy examples remain the core progression.

```{index} Abstract Classes Require an Implementation
```

## Abstract Classes Require an Implementation




```{index} Abstract Classes Require an Implementation
```


An **abstract class** cannot be directly instantiated. It can contain shared implementation and abstract members that concrete derived classes must implement. Use it when a general type has useful common behavior but has no sensible complete instance by itself.

The following payment example gives every payment a validated amount. The abstract operation produces a description that each concrete kind supplies. The array demonstrates the same runtime-dispatch idea as the policy loop.


In [1]:
using System;

Payment[] payments = { new CardPayment(30m, "1234"), new CashPayment(12m) };
foreach (Payment payment in payments) Console.WriteLine(payment.Describe());

abstract class Payment
{
    public decimal Amount { get; }
    protected Payment(decimal amount)
    {
        if (amount < 0m) throw new ArgumentOutOfRangeException(nameof(amount));
        Amount = amount;
    }
    public abstract string Describe();
}
class CardPayment : Payment
{
    private readonly string lastFour;
    public CardPayment(decimal amount, string lastFour) : base(amount)
    {
        if (lastFour.Length != 4) throw new ArgumentException("Four-character teaching ID required.");
        this.lastFour = lastFour;
    }
    public override string Describe() { return $"Card ending {lastFour}: {Amount:F2}"; }
}
class CashPayment : Payment
{
    public CashPayment(decimal amount) : base(amount) { }
    public override string Describe() { return $"Cash: {Amount:F2}"; }
}


Card ending 1234: 30.00
Cash: 12.00


`protected` permits a derived type to call the base constructor while preventing client code from constructing the abstract base directly. The abstract class can own state and concrete methods. The derived `Describe` implementations use `override`, just as with virtual methods.

The short card label is invented teaching data, not a payment-processing system. The model stores no full payment credentials. In a real system the responsibilities for authorizing, charging, and refunding would require explicit contracts and services beyond this chapter.


```{index} Check Every Implementation Against Shared Expectations
```

## Check Every Implementation Against Shared Expectations




```{index} Check Every Implementation Against Shared Expectations
```


Use the same boundary cases for every implementation: zero, a normal subtotal, a subtotal smaller than a fixed reduction, and a negative input. Check that configured rates and reductions are valid too. Testing only through each concrete class can overlook how the client sees the general contract.

A subtype should not introduce hidden output, mutate unrelated state, or quietly change units from cents to dollars. Type compatibility alone cannot protect those meanings. Explain the result interval and side-effect policy in the interface documentation or base-class contract.

The next section removes the need for a shared implementation. An interface describes a capability that unrelated classes can implement. This is useful when inheritance would impose a relationship the business model does not need.


```{index} Practice: Test Through the Base Type
```

## Practice: Test Through the Base Type




```{index} Practice: Test Through the Base Type
```


Run the same result-range check through three base-type references. Include a full percentage reduction and a fixed reduction larger than the subtotal.


In [ ]:
using System;

DiscountPolicy[] policies = {new DiscountPolicy(), new PercentageDiscount(1m), new FixedDiscount(5m)};
foreach (DiscountPolicy policy in policies)
{
    decimal amount = policy.Apply(3m);
    // Your code starts here.
    /* TODO: reject a result outside [0,3]. */
    // Your code ends here.
}
Console.WriteLine("3 substitution checks passed.");

class DiscountPolicy
{
    public virtual decimal Apply(decimal subtotal)
    {
        if (subtotal < 0m) throw new ArgumentOutOfRangeException(nameof(subtotal));
        return subtotal;
    }
}

class PercentageDiscount : DiscountPolicy
{
    private readonly decimal rate;
    public PercentageDiscount(decimal rate)
    {
        if (rate < 0m || rate > 1m) throw new ArgumentOutOfRangeException(nameof(rate));
        this.rate = rate;
    }
    public override decimal Apply(decimal subtotal)
    {
        decimal validSubtotal = base.Apply(subtotal);
        return validSubtotal * (1m - rate);
    }
}

class FixedDiscount : DiscountPolicy
{
    private readonly decimal reduction;
    public FixedDiscount(decimal reduction)
    {
        if (reduction < 0m) throw new ArgumentOutOfRangeException(nameof(reduction));
        this.reduction = reduction;
    }
    public override decimal Apply(decimal subtotal)
    {
        decimal validSubtotal = base.Apply(subtotal);
        return Math.Max(0m, validSubtotal - reduction);
    }
}


In [1]:
using System;

DiscountPolicy[] policies = {new DiscountPolicy(), new PercentageDiscount(1m), new FixedDiscount(5m)};
foreach (DiscountPolicy policy in policies)
{
    decimal amount = policy.Apply(3m);
    if (amount < 0m || amount > 3m) throw new Exception("Contract failed.");
}
Console.WriteLine("3 substitution checks passed.");

class DiscountPolicy
{
    public virtual decimal Apply(decimal subtotal)
    {
        if (subtotal < 0m) throw new ArgumentOutOfRangeException(nameof(subtotal));
        return subtotal;
    }
}

class PercentageDiscount : DiscountPolicy
{
    private readonly decimal rate;
    public PercentageDiscount(decimal rate)
    {
        if (rate < 0m || rate > 1m) throw new ArgumentOutOfRangeException(nameof(rate));
        this.rate = rate;
    }
    public override decimal Apply(decimal subtotal)
    {
        decimal validSubtotal = base.Apply(subtotal);
        return validSubtotal * (1m - rate);
    }
}

class FixedDiscount : DiscountPolicy
{
    private readonly decimal reduction;
    public FixedDiscount(decimal reduction)
    {
        if (reduction < 0m) throw new ArgumentOutOfRangeException(nameof(reduction));
        this.reduction = reduction;
    }
    public override decimal Apply(decimal subtotal)
    {
        decimal validSubtotal = base.Apply(subtotal);
        return Math.Max(0m, validSubtotal - reduction);
    }
}


3 substitution checks passed.


```{rubric} Footnotes
```


[^1202_polymorphism-1]: Overload selection and runtime override dispatch are different language mechanisms. This chapter uses polymorphism to mean calling implementations through a common base type or interface.


[^development-reading]: Further reading: [Microsoft, Methods in C#](https://learn.microsoft.com/en-us/dotnet/csharp/methods). The worked business scenarios and guided reasoning here are original teaching examples.
